# 01 data cleaning

**Owner:** Daniel. Original notebook cells: 5, 56, 57, 58.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

In [ ]:
# Shared project configuration (fixed Databricks volume from the original notebook).
from config.project_config import *
import pandas as pd


In [ ]:
# Daniel: connect to the Team 2 shared Kaggle dataset

import pandas as pd

# Use the shared dataset path configured at the top of the notebook
df = pd.read_csv(NO_SHOW_CSV)

print("Dataset connected:", df.shape)
display(df.head())

### Data Cleaning, Feature Engineering, and Leakage Prevention

In [ ]:
import numpy as np

# 1. Load the raw dataset
appointments = pd.read_csv(NO_SHOW_CSV)

# 2. Data Quality & Cleaning
# Drop anomalous ages (e.g., negative ages)
appointments = appointments[appointments['Age'] >= 0].copy()

# 3. Feature Engineering: Datetimes
# Parse ScheduledDay and AppointmentDay to datetime objects
appointments['ScheduledDay'] = pd.to_datetime(appointments['ScheduledDay'])
appointments['AppointmentDay'] = pd.to_datetime(appointments['AppointmentDay'])

# Create defensible features: Lead Time (days) and Weekday
# Note: AppointmentDay has no time component (00:00:00), so we normalize ScheduledDay's time to midnight for an accurate day difference.
appointments['ScheduledDate'] = appointments['ScheduledDay'].dt.normalize()
appointments['LeadTime_Days'] = (appointments['AppointmentDay'] - appointments['ScheduledDate']).dt.days
appointments['Appointment_Weekday'] = appointments['AppointmentDay'].dt.day_name()

# 4. Filter Logical Anomalies
# Drop records where the appointment happens before it was scheduled (negative lead time)
appointments = appointments[appointments['LeadTime_Days'] >= 0].copy()

# 5. Data Leakage Review
# The 'SMS_received' feature is retained, but noted for Denis's model review. 
# Assumption: SMS is a prior notification. If clinic policy sends SMS *after* a missed appointment, this is severe leakage.
leakage_notes = "Checked SMS_received. Feature is safe assuming it represents pre-appointment reminders. LeadTime_Days engineered safely."
print(leakage_notes)

# 6. Apply Target Encoding (Aligning with Denis's setup)
appointments["target_no_show"] = appointments["No-show"].map({"No": 0, "Yes": 1})

# 7. Export Cleaned Artifact to the shared team 'clean' volume
TEAM_VOLUME = "/Volumes/workspace/default/eye_style_team2"
CLEAN_DATA_DIR = f"{TEAM_VOLUME}/clean"

CLEAN_OUTPUT_PATH = f"{CLEAN_DATA_DIR}/cleaned_appointments_features.csv"
appointments.to_csv(CLEAN_OUTPUT_PATH, index=False)

print(f"Cleaned modeling table saved to: {CLEAN_OUTPUT_PATH}")
print(f"Final usable row count: {len(appointments)}")
display(appointments[['ScheduledDay', 'AppointmentDay', 'LeadTime_Days', 'Appointment_Weekday', 'SMS_received', 'target_no_show']].head())